# Summary Stooq

In [12]:
import duckdb
import pandas as pd
from IPython.display import display

from irp.core.config import config

_con = duckdb.connect(str(config.database.path), read_only=True)

In [13]:
_summary = _con.execute("""
    SELECT
        Src,
        COUNT(*)                              AS rows,
        COUNT(DISTINCT Ticker)                AS tickers,
        MIN(Date)                             AS date_min,
        MAX(Date)                             AS date_max
    FROM prices
    GROUP BY Src
    ORDER BY Src
""").df()
print(f'Total rows : {_summary["rows"].sum():,}')
print(f'Total tickers : {_con.execute("SELECT COUNT(DISTINCT Ticker) FROM prices").fetchone()[0]:,}')
print()
display(_summary)

Total rows : 45,514,486
Total tickers : 14,367



,Src,rows,tickers,date_min,date_max
0,stooq,45514486,14367,17890501,20260424


## Missing Fields

In [14]:
display(_con.execute("""
    SELECT
        Src,
        COUNT(*) FILTER (WHERE O IS NULL) AS missing_O,
        COUNT(*) FILTER (WHERE H IS NULL) AS missing_H,
        COUNT(*) FILTER (WHERE L IS NULL) AS missing_L,
        COUNT(*) FILTER (WHERE C IS NULL) AS missing_C,
        COUNT(*) FILTER (WHERE V IS NULL) AS missing_V
    FROM prices
    GROUP BY Src
    ORDER BY Src
""").df())

,Src,missing_O,missing_H,missing_L,missing_C,missing_V
0,stooq,0,0,0,0,0


## Negative Prices

In [15]:
_neg = _con.execute("""
    SELECT p.Ticker, p.Src, m.Market, p.Date, p.O, p.H, p.L, p.C
    FROM prices p
    LEFT JOIN markets m ON p.Ticker = m.Ticker AND p.Src = m.Src
    WHERE p.C < 0 OR p.O < 0 OR p.H < 0 OR p.L < 0
    ORDER BY p.Ticker, p.Date
""").df()
print(f'Rows with negative prices: {len(_neg):,}  |  Tickers: {_neg["Ticker"].nunique()}')
print()
print('By market:')
display(_neg.groupby('Market', dropna=False).agg(tickers=('Ticker', 'nunique'), rows=('Ticker', 'count')).sort_values('tickers', ascending=False))
if len(_neg):
    display(_neg.head(20))

Rows with negative prices: 128,545  |  Tickers: 121

By market:


,tickers,rows
Market,,
NaN,121,128545


,Ticker,Src,Market,Date,O,H,L,C
0,10YATY,stooq,None,20190618,0.050,0.050,-0.050,-0.044
1,10YATY,stooq,None,20190619,-0.026,0.010,-0.037,-0.005
2,10YATY,stooq,None,20190620,-0.037,-0.011,-0.038,-0.036
3,10YATY,stooq,None,20190621,-0.026,0.019,-0.027,-0.001
4,10YATY,stooq,None,20190624,-0.006,-0.001,-0.032,-0.019
5,10YATY,stooq,None,20190625,-0.033,-0.015,-0.052,-0.046
6,10YATY,stooq,None,20190626,-0.042,-0.013,-0.049,-0.022
7,10YATY,stooq,None,20190627,0.002,0.007,-0.044,-0.041
8,10YATY,stooq,None,20190628,-0.031,-0.028,-0.056,-0.051
9,10YATY,stooq,None,20190701,-0.041,-0.034,-0.091,-0.080


## Stock vs Non-Stock Tickers

In [16]:
_coverage = _con.execute("""
    SELECT
        COUNT(DISTINCT p.Ticker) FILTER (WHERE c.Ticker IS NOT NULL) AS stock_tickers,
        COUNT(DISTINCT p.Ticker) FILTER (WHERE c.Ticker IS NULL)     AS non_stock_tickers
    FROM prices p
    LEFT JOIN companies c ON p.Ticker = c.Ticker
""").df()
display(_coverage)
print('Non-stock tickers include interest rates, FX, crypto, bonds, indices (Stooq coverage is broader than equities).')

,stock_tickers,non_stock_tickers
0,4104,10263


Non-stock tickers include interest rates, FX, crypto, bonds, indices (Stooq coverage is broader than equities).


## Markets

In [ ]:
display(_con.execute("""
    SELECT
        Market,
        COUNT(DISTINCT Ticker) AS tickers
    FROM markets
    GROUP BY Market
    ORDER BY tickers DESC
""").df())